In [ ]:
# Before making any API calls, we need to install the required dependencies
!uv add anthropic python-dotenv

In [ ]:
# Load environment variables (ANTHROPIC_API_KEY) from .env
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from anthropic import Anthropic
from anthropic.types import MessageParam

client = Anthropic()

In [ ]:
# The API is stateless: each call sees only the message we send now.
# So the follow-up below has no memory of the first exchange.

def ask(
    message: str,
    *,
    model: str = "claude-haiku-4-5",
    max_tokens: int = 1000) -> str:
    """Send a single message and return the model's text reply.

    model: which Claude model to use; trades off capability, speed, and cost.
    max_tokens: hard cap on tokens generated — a limit, not a target. If the
        reply would exceed it, output is truncated mid-text and the response's
        stop_reason is "max_tokens" instead of "end_turn".
    """

    # The core of making API requests
    response = client.messages.create(
        model=model, # name of the Claude model we want to use
        messages=[MessageParam(role="user", content=message)], # input to the model
        max_tokens=max_tokens # hard cap on model output
    )

    return next(block.text for block in response.content if block.type == "text")


# Send a message to get the answer
answer = ask("Define quantum computing in one sentence.")
print(answer)

# Follow-up in a separate call — note it has no context from the first
answer = ask("Elaborate further in one sentence.")
print(f"\n{answer}")

In [ ]:
# Helpers to track history ourselves: append each message, then send the whole list.

def add_user_message(messages: list[MessageParam], text: str) -> None:
    messages.append(MessageParam(role="user", content=text))

def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    messages.append(MessageParam(role="assistant", content=text))

def chat(
    messages: list[MessageParam],
    *,
    model: str = "claude-haiku-4-5",
    max_tokens: int = 1000) -> str:
    """Send the full message history and return the reply.

    Unlike ask(), this takes the whole conversation, so the model has context.
    model: which Claude model to use; trades off capability, speed, and cost.
    max_tokens: hard cap on generated tokens — a limit, not a target (truncates).
    """
    response = client.messages.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens
    )

    return next(block.text for block in response.content if block.type == "text")

In [ ]:
# Conversation history
messages: list[MessageParam] = []

# Initial message
add_user_message(messages, "Define quantum computing in one sentence.")

# Send the full history to get an answer
answer = chat(messages)
print(answer)

# Append the answer so the follow-up message has context
add_assistant_message(messages, answer)

# Follow-up message
add_user_message(messages, "Elaborate further in one sentence.")

# Send again - now the follow-up has context
answer = chat(messages)
print(f"\n{answer}")

In [ ]:
# Interactive multi-turn chat: type messages, "stop" to quit.
messages: list[MessageParam] = []

while True:
    user_message = input()
    print(f"user: {user_message}")

    if user_message == "stop":
        print("user stopped the session")
        break

    add_user_message(messages, user_message)

    assistant_message = chat(messages)
    print(f"assistant: {assistant_message}")

    add_assistant_message(messages, assistant_message)